# BÀI THỰC HÀNH MÁY HỌC: QUY TRÌNH TIỀN XỬ LÝ & HỒI QUY TUYẾN TÍNH
**Môn học:** Nhập môn Học máy / Machine Learning  
**Hình thức làm bài:** Cá nhân  
**Thời gian:** 90 phút  

---
### THÔNG TIN SINH VIÊN
- **Họ và tên:** *[Phạm Nguyễn Phúc Khang]*
- **Mã số sinh viên (MSSV):** *[24646421]*
- **Lớp:** *[DHKHMT20A]*
- **Ngày thực hiện:** *[09/04/2026]*

---
### HƯỚNG DẪN LÀM BÀI:
1. Sinh viên điền đúng MSSV vào **Phần 0** để khởi tạo tập dữ liệu được cá nhân hóa.
2. Hoàn thành tất cả các mục có đánh dấu `# TODO: Sinh viên viết code tại đây`.
3. Chạy lại toàn bộ notebook (`Kernel -> Restart and Run All`) trước khi nộp để đảm bảo không có lỗi runtime.
4. Nộp lại file `.ipynb` theo định dạng <MSSV_hotenkhongdauvietlien>.ipynb.

## 0. KHỞI TẠO TẬP DỮ LIỆU CÁ NHÂN HÓA (Personalized Data Generation)
Chạy đoạn mã dưới đây để sinh dữ liệu ngẫu nhiên theo hạt giống (seed) là MSSV của bạn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cài đặt hiển thị đồ thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# ====================================================================
# ĐIỀN MSSV CỦA BẠN VÀO ĐÂY (VÍ DỤ: 20210123)
# ====================================================================
MSSV = 24646421  # <-- THAY ĐỔI GIÁ TRỊ NÀY

np.random.seed(MSSV)
n = 1200

# 1. Biến liên tục
x_area = np.random.uniform(30, 250, n)  # Diện tích (m2)
x_dist = np.random.exponential(scale=5, size=n)  # Khoảng cách đến TT (km)
x_age = np.random.randint(0, 30, n)  # Tuổi thọ công trình (năm)

# 2. Biến phân loại (Categorical có format bị nhiễu)
loc_types = ['Trung tâm', 'trung tam', 'Ngoại thành', 'ngoai_thanh', 'Ven đô', 'Ven Do', 'UNKNOWN']
x_loc = np.random.choice(loc_types, size=n, p=[0.25, 0.05, 0.35, 0.05, 0.20, 0.05, 0.05])

# 3. Target y (Giá nhà - triệu VNĐ) + Nhiễu phi tuyến và ngoại lai (Outliers)
loc_effect = {
    'Trung tâm': 1.5, 'trung tam': 1.5, 
    'Ngoại thành': 0.8, 'ngoai_thanh': 0.8, 
    'Ven đô': 1.0, 'Ven Do': 1.0, 
    'UNKNOWN': 1.0
}
loc_mult = np.array([loc_effect[k] for k in x_loc])
y = (500 + 35 * x_area - 25 * x_dist - 8 * x_age + 0.05 * (x_area ** 1.3)) * loc_mult + np.random.normal(0, 50, n)

# 4. Gài bẫy Outliers & Missing Values
x_age[np.random.choice(n, 15, replace=False)] = -99   # Bẫy missing ngầm định
y[np.random.choice(n, 10, replace=False)] = -500      # Bẫy giá trị âm vô lý

df_raw = pd.DataFrame({
    'DienTich': x_area, 
    'KhoangCach': x_dist, 
    'TuoiTho': x_age, 
    'KhuVuc': x_loc, 
    'GiaNha': y
})
df_raw.loc[np.random.choice(n, 30, replace=False), 'DienTich'] = np.nan
df_raw.loc[np.random.choice(n, 20, replace=False), 'KhoangCach'] = -1.0 # Bẫy khoảng cách âm

print(f"[*] Đã khởi tạo thành công tập dữ liệu cho MSSV: {MSSV}")
print(f"[*] Kích thước dữ liệu ban đầu: {df_raw.shape}")
df_raw.head(10)

: 

--- 
## CÂU 1: KHÁM PHÁ & TIỀN XỬ LÝ DỮ LIỆU (3.0 Điểm)

### 1.1 Khám phá dữ liệu & Xác định các bất thường
- Xem thống kê mô tả (`.describe()`, `.info()`).
- Đếm số lượng giá trị `NaN`, giá trị âm hoặc giá trị đặc biệt (`-99`, `-1.0`, `UNKNOWN`).

In [ ]:
# TODO: Sinh viên viết code thống kê mô tả và đếm các giá trị bất thường tại đây
display(df_raw.describe()) # describe
display(df_raw.info()) # info
display(df_raw.isna().sum()) # tổng NaN
display((df_raw['TuoiTho'] == -99).sum())
display((df_raw['KhoangCach'] == -1.0).sum())
display((df_raw['DienTich'] == "UNKNOWN").sum())
display((df_raw['GiaNha'] < 0).sum())


: 

### 1.2 Xử lý giá trị phi logic & Khuyết thiếu (Missing Values)
- **Quy tắc:**
  1. `TuoiTho == -99`: Đây là mã hóa missing ngầm định -> chuyển thành `np.nan` rồi điền bằng giá trị **Median** của tập dữ liệu.
  2. `KhoangCach == -1.0`: Giá trị khoảng cách không thể âm -> chuyển thành `np.nan` rồi điền bằng giá trị **Median**.
  3. `DienTich` bị `NaN`: Điền bằng giá trị **Mean** hoặc **Median**.
  4. `GiaNha <= 0`: Giá nhà không thể âm hoặc bằng 0 -> **Xóa bỏ các dòng này** khỏi tập dữ liệu.

In [ ]:
# TODO: Sinh viên thực hiện làm sạch các giá trị phi logic và điền khuyết thiếu
df_clean = df_raw.copy()
# Viết code tại đây...
df_clean['TuoiTho'] = df_clean['TuoiTho'].replace(-99,np.nan)
df_clean['TuoiTho'] = df_clean['TuoiTho'].fillna(df_clean['TuoiTho'].median())

df_clean['KhoangCach'] = df_clean['KhoangCach'].replace(-1.0,np.nan)
df_clean['KhoangCach'] = df_clean['KhoangCach'].fillna(df_clean['KhoangCach'].median())

df_clean['DienTich'] = df_clean['DienTich'].fillna(df_clean['DienTich'].median())
df_clean = df_clean[(df_clean['GiaNha'] > 0)].reset_index(drop=True)

: 

### 1.3 Chuẩn hóa biến phân loại (`KhuVuc`)
- Gom nhóm chuẩn hóa thành 3 nhóm duy nhất:
  - `['Trung tâm', 'trung tam']` $\rightarrow$ `'Trung tam'`
  - `['Ngoại thành', 'ngoai_thanh']` $\rightarrow$ `'Ngoai thanh'`
  - `['Ven đô', 'Ven Do']` $\rightarrow$ `'Ven do'`
- Với giá trị `'UNKNOWN'`: Thay thế bằng nhóm xuất hiện nhiều nhất (Mode) hoặc nhóm mặc định `'Ven do'`.
- Hiển thị bảng tần suất sau khi làm sạch.

In [ ]:
# TODO: Sinh viên viết code chuẩn hóa chuỗi text cho cột 'KhuVuc'
def chuanHoa(val):
    if val in ["Trung tâm", "trung tam"]:
        return "Trung tam"
    elif val in ["Ngoại thành", 'ngoai_thanh']:
        return  'Ngoai thanh'
    else:
        return "Ven do"
df_clean['KhuVuc'] = df_clean['KhuVuc'].apply(chuanHoa)
display(df_clean['KhuVuc'])
print(df_clean["KhuVuc"].value_counts())

: 

### 1.4 Phát hiện & Lọc Outliers trên biến mục tiêu (`GiaNha`)
- Sử dụng phương pháp **IQR (Interquartile Range)** để tìm ngưỡng trên (`Q3 + 1.5 * IQR`) và ngưỡng dưới (`Q1 - 1.5 * IQR`).
- Loại bỏ các điểm ngoại lai.
- Vẽ biểu đồ Boxplot của `GiaNha` **trước** và **sau** khi lọc Outliers (đặt cạnh nhau bằng subplot).

In [ ]:
Q1 = df_clean['GiaNha'].quantile(0.25)
Q3 = df_clean['GiaNha'].quantile(0.75)
IQR = Q3 - Q1
Above = Q3 + 1.5 * IQR
Below = Q1 - 1.5 * IQR
fig, axes = plt.subplots(1,2,figsize=(12,5))

sns.boxplot(data=df_clean['GiaNha'],ax = axes[0], color="orange")
axes[0].set_title('Boxplot gia nha truoc khi loc outliers')

df_clean = df_clean[(df_clean['GiaNha'] >= Below) & (df_clean['GiaNha'] <= Above)]
sns.boxplot(data=df_clean['GiaNha'],ax = axes[1], color="skyblue")
axes[1].set_title('Boxplot gia nha sau khi loc outliers')
plt.tight_layout()
plt.show()

: 

--- 
## CÂU 2: THIẾT KẾ LUỒNG MÁY HỌC & NGĂN CHẶN RÒ RỈ DỮ LIỆU (3.0 Điểm)

### 2.1 Phân chia tập dữ liệu (Train / Validation / Test)
- Tách biến đặc trưng $X$ và biến mục tiêu $y$.
- Chia dữ liệu theo tỷ lệ **70% Train, 15% Validation, 15% Test**.
- Thiết lập `random_state = MSSV % 100` để đảm bảo tính tái lập.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: Sinh viên thực hiện chia tập dữ liệu thành (X_train, y_train), (X_val, y_val), (X_test, y_test)
MSSV = 24646421
seed = MSSV % 100
X = df_clean.drop(columns=["GiaNha"])
y = df_clean["GiaNha"]
# Viết code tại đây...
X_train_val, X_test, y_train_val, y_test = train_test_split(X,y,test_size=0.15,random_state=seed)
X_train,X_val,y_train,y_val=train_test_split(X_train_val,y_train_val,test_size=(0.15/0.85),random_state=seed)

print(f"Kich thuoc tap Train: {X_train.shape}")
print(f"Kich thuoc tap Train: {X_val.shape}")
print(f"Kich thuoc tap Train: {X_test.shape}")

: 

### 2.2 Xây dựng tiền xử lý chuẩn hóa & Mã hóa (Tránh Data Leakage)
- Sử dụng `ColumnTransformer` hoặc `Pipeline` từ `sklearn`:
  - Các cột số (`DienTich`, `KhoangCach`, `TuoiTho`): Áp dụng `StandardScaler`.
  - Cột phân loại (`KhuVuc`): Áp dụng `OneHotEncoder(drop='first')`.
- **Lưu ý quan trọng:** Chỉ gọi `.fit()` hoặc `.fit_transform()` trên tập **Train**, sau đó dùng `.transform()` trên tập **Val** và **Test**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: Sinh viên định nghĩa preprocessor và biến đổi X_train, X_val, X_test

num_f = ['DienTich','KhoangCach','TuoiTho']
cat_f = ['KhuVuc']
preprocessor = ColumnTransformer(transformers=[('num',StandardScaler(),num_f),
                                               ('cat',OneHotEncoder(drop='first',handle_unknown='ignore'),cat_f)])
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.fit_transform(X_val)
X_test_processed = preprocessor.fit_transform(X_test)

: 

--- 
## CÂU 3: HUẤN LUYỆN MÔ HÌNH HỒI QUY & TỐI ƯU HÓA (2.5 Điểm)

### 3.1 Mô hình Hồi quy tuyến tính chuẩn (Ordinary Least Squares - OLS)
- Huấn luyện `LinearRegression` trên tập Train.
- In ra phương trình hồi quy (Intercept $w_0$ và các Coefficients $w_i$ tương ứng từng đặc trưng).

In [ ]:
from sklearn.linear_model import LinearRegression

# TODO: Huấn luyện Linear Regression và hiển thị trọng số w
ols_model = LinearRegression()
ols_model.fit(X_train_processed,y_train)
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(cat_f))
all_features = num_f + encoded_cat_features
print(f"Intercept (w_0): {ols_model.intercept_:.4f}")
print("Trong so:")
for feat,coef in zip(all_features, ols_model.coef_):
    print(f"{feat}:{coef:.4f}")

: 

### 3.2 Mô hình Hồi quy Ridge với K-Fold Cross-Validation ($K=5$)
- Sử dụng `RidgeCV` hoặc `GridSearchCV` với $K=5$ folds trên tập Train để tìm hệ số $\alpha$ tối ưu từ danh sách `alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]`.
- In ra giá trị $\alpha$ tốt nhất tìm được.

In [ ]:
from sklearn.linear_model import RidgeCV, Ridge

# TODO: Tìm alpha tối ưu cho Ridge Regression bằng Cross-Validation
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_cv = RidgeCV(alphas=alphas,cv=5,scoring='r2')
ridge_cv.fit(X_train_processed,y_train)
best_alpha=ridge_cv.alpha_
print(f"gia tri toi uu {best_alpha}")
ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train_processed,y_train)

: 

### 3.3 Đánh giá và so sánh hiệu năng ($R^2$, $MAE$, $RMSE$)
- Tính toán $R^2$, $MAE$, $RMSE$ cho cả **Linear Regression** và **Ridge Regression** trên cả 3 tập: Train, Validation, Test.
- Trình bày kết quả dưới dạng một bảng `pd.DataFrame` trực quan.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# TODO: Tính toán các độ đo và hiển thị bảng so sánh
def evaluate_model(model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    res = {}
    for name, X_data, y_data in [('Train', X_tr, y_tr), ('Validation', X_v, y_v), ('Test', X_te, y_te)]:
        y_pred = model.predict(X_data)
        res[f'R2_{name}'] = r2_score(y_data,y_pred)
        res[f'MAE_{name}'] = mean_absolute_error(y_data,y_pred)
        res[f'RMSE_{name}'] = np.sqrt(mean_squared_error(y_data,y_pred))
    return res
ols_metrics = evaluate_model(ols_model,X_train_processed,y_train,X_val_processed,y_val,X_test_processed,y_test)
ridge_metrics = evaluate_model(ridge_model,X_train_processed,y_train,X_val_processed,y_val,X_test_processed,y_test)
df_comparison = pd.DataFrame([ols_metrics,ridge_metrics],index=["Linear Regression (OLS)", "Ridge Regression"])
display(df_comparison.T)

: 

### 3.4 Vẽ biểu đồ Phần dư (Residual Plot)
- Tính phần dư trên tập Test: $e = y_{test} - \hat{y}_{test}$.
- Vẽ biểu đồ phân tán: Trục hoành là $\hat{y}_{test}$ (Giá trị dự đoán), trục tung là Phần dư $e$.
- Vẽ đường ngang $y = 0$ màu đỏ để đối chiếu.

In [ ]:
# TODO: Vẽ Residual Plot trên tập Test

y_pred_test = ridge_model.predict(X_test_processed)
residuals = y_test - y_pred_test
plt.figure(figsize=(8,5))
plt.scatter(y_pred_test,residuals,alpha=0.6,color="b")
plt.axhline(y=0,color="r",linestyle="--",linewidth=2)
plt.xlabel('Gia tri du doan y_pred')
plt.ylabel("Phan du Residuals: y_true - y_pred")
plt.title("Residual Plot tren test (Ridge Reg)")
plt.grid(True)
plt.show()

: 

--- 
## CÂU 4: PHÂN TÍCH & TRẢ LỜI CÂU HỎI CHUYÊN SÂU (1.5 Điểm)
*(Sinh viên viết câu trả lời trực tiếp vào các ô Markdown bên dưới)*

### Câu hỏi 4.1: Hiện tượng trên biểu đồ phần dư (Residual Plot)
Quan sát biểu đồ phần dư bạn vừa vẽ ở mục 3.4: Các điểm phần dư phân tán ngẫu nhiên đồng đều (Homoscedasticity) hay có dạng hình phễu / đường cong (Heteroscedasticity)? Hiện tượng này giải thích điều gì về bản chất mối quan hệ thực tế giữa các thuộc tính và giá nhà trong tập dữ liệu của bạn?

**Trả lời của sinh viên:**  
>[Dựa vào biểu đồ phần dư, thì các điểm không phân tán ngẫu nhiên xung quanh y = 0 mà co xu hướng mở rộng biên độ dao động khi giá trị dự đoán tăng dần => Hiện tượng phương sai sai số thay đổi

>Hiện tượng này cho thấy mối quan hệ giữa các thuộc tính và giá nhà trong tập dữ liệu mang tính chất phi tuyến tính (Mô hình Linear hiện tại chưa giải thích hết biến động dữ liệu)

>Ở phân khúc giá nhà cao, độ lệch giữa giá thực tế và giá trị dự đoán lớn hơn rất nhiều so với phân khúc giá thấp. Điều này phản ánh thực tế là các bất động sản cao cấp thường có mức độ biến động giá phức tạp hon và khó dự đoán chính xác chỉ dựa vào các thuộc tính tuyến tính cơ bản]
*

---
### Câu hỏi 4.2: Phân tích ý nghĩa trọng số (Feature Importance)
So sánh trọng số giữa OLS và Ridge. Biến đặc trưng nào có ảnh hưởng mạnh nhất đến việc tăng hoặc giảm giá nhà? Ý nghĩa kinh tế/thực tế của trọng số đó là gì?

**Trả lời của sinh viên:**  
>*[So sánh trọng số giữa OLS và Ridge: 
>- Trọng số của OLS có xu hướng lớn hơn và dao động mạnh do nhạy cảm với hiện tượng đa cộng tuyến trong tập dũ liệu bất động sản
>- Biến đặc trưng nào có ảnh hưởng mạnh nhất đến việc tăng hoặc giảm giá nhà?
>- Cách tìm: 
>- Điều kiện tiên quyết: Dữ liệu đầu vào phải được chuẩn hóa trước khi chạy Ridge.
>- Cách xác định biến tăng mạnh nhất: Tìm biến có hệ số hồi quy mang dấu + và có giá trị tuyết đối lớn nhất
>- Cách xác định biến giảm mạnh nhất: Tìm biến có hệ số hồi quy mang dấu - và có giá trị tuyết đối lớn nhất
>- Ý nghĩa kinh tế: Giá nhà sẽ thay đổi một lượng đúng bằng giá trị của trong số tương ứng trong điều kiện các yếu tố khác không đổi
>- Ý nghĩa thực tế: Thể hiện mức độ khẩu hao tài sản vật chất theo thời gian. Nhà càng cũ thì chi phí càng cao, làm giảm giá trị giao dịch trên thị trường kinh tế.]*

---
### Câu hỏi 4.3: Vấn đề ngoại suy (Extrapolation Limitations)
Nếu có một khách hàng muốn dự đoán giá một căn biệt thự diện tích $600m^2$ (vượt xa khoảng giá trị 30 - 250 $m^2$ trong tập dữ liệu huấn luyện), mô hình hồi quy tuyến tính của bạn có thể đưa ra kết quả tin cậy không? Vì sao?

**Trả lời của sinh viên:**  
>*[Vì những lý do sau:
>- Vấn đề ngoại suy: Diện tích 600m vuông nằm ngoài khoảng dữ liệu huấn luyện (30 - 250m vuông). Mô hình tuyến tính giá định rằng mối quan hệ giữa diện tích và giá nhà là một đường thẳng cố định dựa trên tập data cũ, nhưng không có gì đảm bảo mối quan hệ này vẫn đúng ở các giá trị nằm ngoài khoảng đó 
>- Thiếu dũ liệu kiểm chứng: Do dữ liệu chưa đủ ngoài 600m vuông nên mô hình chỉ mang tính chất kéo dài đường thẳng lý thuyết và có sai số to
>- Thay đổi về quy luật thực tế: Biệt thự lớn với 600m vuông thì có thể có giá trị tăng nhanh nếu ở phân khúc siêu sang/hiếm và chậm do biên lợi nhuận giảm hoặc kém người mua]*